# MNIST Neural Network: Beginner Level 2
## Single Class (Monolith) Approach

**Complexity Level**: Beginner  
**Architectural Style**: Single class encapsulating everything  
**Focus**: Introducing OOP concepts while keeping the structure simple

---

## The Problem: Handwritten Digit Recognition

The MNIST dataset presents a classic machine learning challenge: given an image of a handwritten digit, determine which digit (0-9) it represents.

**Inputs**: Each image is a 28×28 pixel grayscale image, which we flatten into a single list of 784 values. Each pixel value ranges from 0 (black) to 255 (white), though we typically normalise these to the range 0.0 to 1.0 by dividing by 255. So our neural network receives 784 floating-point numbers as input for each image.

**Outputs**: The network produces 10 values, one for each possible digit (0 through 9). These outputs represent the network's confidence that the input image shows each respective digit. We typically interpret these as probabilities (summing to 1.0 after applying softmax), and the digit with the highest probability becomes our prediction. For training, we compare these outputs against "one-hot" encoded labels where the correct digit has value 1.0 and all others have 0.0.

**The Task**: Build a function that maps 784 input values to 10 output values, learning the patterns that distinguish each digit through repeated exposure to labelled examples.

---

## Why This Approach?

This notebook wraps everything in a **single `NeuralNetwork` class**. This approach:

- Encapsulates all network state (weights, biases) in one object
- Provides a clean interface: `network.forward()`, `network.train()`, `network.predict()`
- Introduces basic OOP without multiple interacting classes
- Keeps implementation details private inside the class

**Trade-offs**:
- The class does many things (violates Single Responsibility Principle)
- Harder to swap out components (e.g., different optimisers)
- All complexity hidden in one place

**Alternative**: Could split into Network, Trainer, and Evaluator classes (see intermediate notebooks).

In [ ]:
# Only standard library imports
import math
import random

# Set seed for reproducibility
random.seed(42)

---

## The Complete Neural Network Class

This single class contains:
- Network architecture and parameters (weights, biases)
- Activation functions (as static methods)
- Forward pass computation
- Backpropagation and training
- Evaluation and prediction

**Design Choice**: Activation functions are implemented as static methods within the class.

**Alternative**: Could use a separate dictionary mapping names to functions, or pass activation functions as constructor arguments.

In [ ]:
class NeuralNetwork:
    """
    A complete feedforward neural network in a single class.
    
    This 'monolith' approach keeps everything together, making it easy to
    understand the complete picture. However, it violates the Single
    Responsibility Principle - this class does network structure, training,
    and evaluation all at once.
    
    Attributes:
        layer_sizes: Architecture as list of integers
        weights: Nested list of weight matrices
        biases: List of bias vectors
        learning_rate: Step size for gradient descent
        hidden_activation: Which activation to use ('relu' or 'sigmoid')
    """
    
    def __init__(self, layer_sizes, learning_rate=0.01, hidden_activation='relu'):
        """
        Initialise network with given architecture.
        
        Args:
            layer_sizes: List like [784, 128, 10] defining structure
            learning_rate: How fast to learn (higher = faster but less stable)
            hidden_activation: 'relu' (default) or 'sigmoid'
        
        Design choice: We store layer_sizes to allow architecture inspection.
        Alternative: Just store weights/biases, infer sizes from shapes.
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.hidden_activation = hidden_activation
        
        # Initialise parameters
        self.weights = []
        self.biases = []
        self._initialise_parameters()
    
    def _initialise_parameters(self):
        """
        Create and initialise all weights and biases.
        
        Uses Xavier initialisation: weights ~ uniform(-limit, limit)
        where limit = sqrt(6 / (fan_in + fan_out))
        
        Alternative: He initialisation (better for ReLU), or simple random.
        
        The underscore prefix indicates this is a 'private' method - it's
        meant to be called internally, not by users of the class.
        """
        for i in range(len(self.layer_sizes) - 1):
            fan_in = self.layer_sizes[i]
            fan_out = self.layer_sizes[i + 1]
            
            # Xavier initialisation
            limit = math.sqrt(6.0 / (fan_in + fan_out))
            
            # Create weight matrix
            layer_weights = [
                [random.uniform(-limit, limit) for _ in range(fan_out)]
                for _ in range(fan_in)
            ]
            self.weights.append(layer_weights)
            
            # Create bias vector (initialised to zero)
            self.biases.append([0.0] * fan_out)
    
    # ==================== Activation Functions ====================
    # These are static methods - they don't use 'self' because they're
    # pure mathematical functions that don't depend on network state.
    
    @staticmethod
    def _sigmoid(x):
        """Sigmoid activation: squashes to (0, 1)."""
        x = max(-500, min(500, x))  # Prevent overflow
        return 1.0 / (1.0 + math.exp(-x))
    
    @staticmethod
    def _sigmoid_derivative(x):
        """Derivative of sigmoid."""
        s = NeuralNetwork._sigmoid(x)
        return s * (1 - s)
    
    @staticmethod
    def _relu(x):
        """ReLU activation: max(0, x)."""
        return max(0.0, x)
    
    @staticmethod
    def _relu_derivative(x):
        """Derivative of ReLU."""
        return 1.0 if x > 0 else 0.0
    
    @staticmethod
    def _softmax(values):
        """Softmax: converts to probability distribution."""
        max_val = max(values)
        exp_values = [math.exp(v - max_val) for v in values]
        total = sum(exp_values)
        return [ev / total for ev in exp_values]
    
    def _get_activation(self):
        """
        Return activation function and derivative based on configuration.
        
        Design choice: Returns tuple of (function, derivative).
        Alternative: Could use a dictionary or strategy pattern.
        """
        if self.hidden_activation == 'sigmoid':
            return self._sigmoid, self._sigmoid_derivative
        else:  # Default to relu
            return self._relu, self._relu_derivative
    
    # ==================== Forward Pass ====================
    
    def forward(self, inputs):
        """
        Compute network output for given inputs.
        
        Args:
            inputs: List of input values (e.g., 784 pixel values)
        
        Returns:
            outputs: Network predictions (probabilities)
            cache: Intermediate values needed for backpropagation
        
        Design choice: Returns cache for training. For prediction-only,
        could have a separate method that doesn't compute cache.
        """
        activate, _ = self._get_activation()
        
        # Cache stores values needed for backpropagation
        cache = {
            'layer_inputs': [inputs],   # Pre-activation values
            'layer_outputs': [inputs]   # Post-activation values
        }
        
        current = inputs
        
        for layer_idx in range(len(self.weights)):
            layer_w = self.weights[layer_idx]
            layer_b = self.biases[layer_idx]
            
            # Compute weighted sums: z = W * x + b
            weighted_sums = []
            for j in range(len(layer_b)):
                total = layer_b[j]
                for i in range(len(current)):
                    total += current[i] * layer_w[i][j]
                weighted_sums.append(total)
            
            cache['layer_inputs'].append(weighted_sums)
            
            # Apply activation
            is_output = (layer_idx == len(self.weights) - 1)
            if is_output:
                current = self._softmax(weighted_sums)
            else:
                current = [activate(x) for x in weighted_sums]
            
            cache['layer_outputs'].append(current)
        
        return current, cache
    
    def predict(self, inputs):
        """
        Make a prediction (returns class index).
        
        This is a convenience method for inference.
        """
        outputs, _ = self.forward(inputs)
        return outputs.index(max(outputs))
    
    def predict_proba(self, inputs):
        """
        Get prediction probabilities.
        
        Returns full probability distribution, not just the class.
        """
        outputs, _ = self.forward(inputs)
        return outputs
    
    # ==================== Loss Function ====================
    
    @staticmethod
    def _cross_entropy_loss(predicted, target):
        """
        Cross-entropy loss for classification.
        
        Lower values indicate better predictions.
        
        Alternative: Could use mean squared error (simpler but worse for classification).
        """
        epsilon = 1e-15
        loss = 0.0
        for p, t in zip(predicted, target):
            if t > 0:
                loss -= t * math.log(max(p, epsilon))
        return loss
    
    @staticmethod
    def _create_one_hot(label, num_classes):
        """Convert integer label to one-hot vector."""
        one_hot = [0.0] * num_classes
        one_hot[label] = 1.0
        return one_hot
    
    # ==================== Backpropagation ====================
    
    def _backward(self, target, cache):
        """
        Compute gradients via backpropagation.
        
        Args:
            target: One-hot encoded true label
            cache: Values from forward pass
        
        Returns:
            weight_grads: Gradients for weights
            bias_grads: Gradients for biases
        
        This implements the chain rule to find how each parameter
        affects the loss.
        """
        _, activation_derivative = self._get_activation()
        
        layer_inputs = cache['layer_inputs']
        layer_outputs = cache['layer_outputs']
        
        # Initialise gradient storage
        weight_grads = [
            [[0.0] * len(self.weights[l][0]) for _ in range(len(self.weights[l]))]
            for l in range(len(self.weights))
        ]
        bias_grads = [[0.0] * len(b) for b in self.biases]
        
        # Output layer error (softmax + cross-entropy derivative)
        output = layer_outputs[-1]
        deltas = [output[i] - target[i] for i in range(len(output))]
        
        # Backpropagate through layers
        for layer_idx in range(len(self.weights) - 1, -1, -1):
            prev_output = layer_outputs[layer_idx]
            
            # Compute gradients
            for j in range(len(deltas)):
                bias_grads[layer_idx][j] = deltas[j]
                for i in range(len(prev_output)):
                    weight_grads[layer_idx][i][j] = prev_output[i] * deltas[j]
            
            # Compute deltas for previous layer
            if layer_idx > 0:
                layer_in = layer_inputs[layer_idx]
                new_deltas = []
                for i in range(len(prev_output)):
                    error_sum = sum(
                        deltas[j] * self.weights[layer_idx][i][j]
                        for j in range(len(deltas))
                    )
                    new_deltas.append(error_sum * activation_derivative(layer_in[i]))
                deltas = new_deltas
        
        return weight_grads, bias_grads
    
    def _update_parameters(self, weight_grads, bias_grads):
        """
        Apply gradient descent update.
        
        Formula: parameter -= learning_rate * gradient
        
        Alternative: Could implement momentum or Adam optimiser here.
        """
        for l in range(len(self.weights)):
            for i in range(len(self.weights[l])):
                for j in range(len(self.weights[l][i])):
                    self.weights[l][i][j] -= self.learning_rate * weight_grads[l][i][j]
            for j in range(len(self.biases[l])):
                self.biases[l][j] -= self.learning_rate * bias_grads[l][j]
    
    # ==================== Training ====================
    
    def train_step(self, inputs, label):
        """
        Train on a single example.
        
        Args:
            inputs: Input features
            label: Integer class label
        
        Returns:
            loss: Loss value for this example
            correct: Whether prediction was correct
        """
        # Forward pass
        outputs, cache = self.forward(inputs)
        
        # Compute loss
        target = self._create_one_hot(label, len(outputs))
        loss = self._cross_entropy_loss(outputs, target)
        
        # Check prediction
        predicted = outputs.index(max(outputs))
        correct = (predicted == label)
        
        # Backward pass
        weight_grads, bias_grads = self._backward(target, cache)
        
        # Update parameters
        self._update_parameters(weight_grads, bias_grads)
        
        return loss, correct
    
    def fit(self, data, labels, epochs=10, verbose=True):
        """
        Train on dataset for multiple epochs.
        
        Args:
            data: List of input vectors
            labels: List of integer labels
            epochs: Number of passes through data
            verbose: Whether to print progress
        
        Returns:
            history: Dictionary with loss and accuracy per epoch
        
        Design choice: Returns history for plotting.
        Alternative: Could use callbacks for custom logging.
        """
        history = {'loss': [], 'accuracy': []}
        
        for epoch in range(epochs):
            total_loss = 0.0
            correct_count = 0
            
            for inputs, label in zip(data, labels):
                loss, correct = self.train_step(inputs, label)
                total_loss += loss
                correct_count += int(correct)
            
            avg_loss = total_loss / len(data)
            accuracy = correct_count / len(data)
            
            history['loss'].append(avg_loss)
            history['accuracy'].append(accuracy)
            
            if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
                print(f"Epoch {epoch+1:3d}: Loss={avg_loss:.4f}, Accuracy={accuracy:.3f}")
        
        return history
    
    # ==================== Evaluation ====================
    
    def evaluate(self, data, labels):
        """
        Evaluate network on test data.
        
        Returns:
            accuracy: Fraction of correct predictions
        """
        correct = sum(
            1 for inputs, label in zip(data, labels)
            if self.predict(inputs) == label
        )
        return correct / len(data)
    
    # ==================== Utility Methods ====================
    
    def summary(self):
        """
        Print network architecture summary.
        """
        print("Neural Network Summary")
        print("=" * 40)
        print(f"Architecture: {' -> '.join(map(str, self.layer_sizes))}")
        print(f"Hidden activation: {self.hidden_activation}")
        print(f"Learning rate: {self.learning_rate}")
        print()
        
        total_params = 0
        for i in range(len(self.weights)):
            w_count = len(self.weights[i]) * len(self.weights[i][0])
            b_count = len(self.biases[i])
            layer_params = w_count + b_count
            total_params += layer_params
            print(f"Layer {i}: {self.layer_sizes[i]} -> {self.layer_sizes[i+1]}")
            print(f"         {w_count} weights + {b_count} biases = {layer_params} params")
        
        print("=" * 40)
        print(f"Total parameters: {total_params:,}")
    
    def get_weights(self):
        """Return copy of weights (for inspection/saving)."""
        return [[[w for w in row] for row in layer] for layer in self.weights]
    
    def set_weights(self, new_weights):
        """Set weights (for loading saved model)."""
        self.weights = [[[w for w in row] for row in layer] for layer in new_weights]

---

## Using the Neural Network

Now we can use our class with a clean, simple interface.

In [ ]:
# Helper function to generate synthetic data
def generate_data(num_samples, input_size, num_classes):
    """
    Generate synthetic classification data.
    Label is based on which portion of input has highest sum.
    """
    data = []
    labels = []
    chunk_size = input_size // num_classes
    
    for _ in range(num_samples):
        inputs = [random.random() for _ in range(input_size)]
        chunk_sums = [
            sum(inputs[c * chunk_size:(c + 1) * chunk_size])
            for c in range(num_classes)
        ]
        label = chunk_sums.index(max(chunk_sums))
        data.append(inputs)
        labels.append(label)
    
    return data, labels

In [ ]:
# Create network
INPUT_SIZE = 20
HIDDEN_SIZE = 16
NUM_CLASSES = 4

network = NeuralNetwork(
    layer_sizes=[INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES],
    learning_rate=0.1,
    hidden_activation='relu'
)

network.summary()

In [ ]:
# Generate data
train_data, train_labels = generate_data(500, INPUT_SIZE, NUM_CLASSES)
test_data, test_labels = generate_data(100, INPUT_SIZE, NUM_CLASSES)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

In [ ]:
# Train the network
print("Training...\n")
history = network.fit(train_data, train_labels, epochs=20, verbose=True)

# Evaluate
test_accuracy = network.evaluate(test_data, test_labels)
print(f"\nFinal test accuracy: {test_accuracy:.3f}")

In [ ]:
# Make some predictions
print("Sample predictions:")
print("-" * 50)

for i in range(5):
    probabilities = network.predict_proba(test_data[i])
    predicted = network.predict(test_data[i])
    actual = test_labels[i]
    status = "Correct" if predicted == actual else "Wrong"
    
    print(f"\nSample {i+1}:")
    print(f"  Predicted: {predicted}, Actual: {actual} ({status})")
    print(f"  Probabilities: {[f'{p:.3f}' for p in probabilities]}")

---

## Comparing Different Configurations

One benefit of the class approach: we can easily create multiple networks with different settings.

In [ ]:
# Compare ReLU vs Sigmoid
print("Comparing activation functions:")
print("=" * 50)

for activation in ['relu', 'sigmoid']:
    # Create fresh network
    net = NeuralNetwork(
        layer_sizes=[INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES],
        learning_rate=0.1,
        hidden_activation=activation
    )
    
    # Train
    net.fit(train_data, train_labels, epochs=15, verbose=False)
    
    # Evaluate
    train_acc = net.evaluate(train_data, train_labels)
    test_acc = net.evaluate(test_data, test_labels)
    
    print(f"{activation.upper():8s}: Train={train_acc:.3f}, Test={test_acc:.3f}")

In [ ]:
# Compare different architectures
print("\nComparing architectures:")
print("=" * 50)

architectures = [
    [INPUT_SIZE, 8, NUM_CLASSES],           # Small
    [INPUT_SIZE, 16, NUM_CLASSES],          # Medium
    [INPUT_SIZE, 32, NUM_CLASSES],          # Large
    [INPUT_SIZE, 16, 8, NUM_CLASSES],       # Two hidden layers
]

for arch in architectures:
    net = NeuralNetwork(arch, learning_rate=0.1)
    net.fit(train_data, train_labels, epochs=15, verbose=False)
    test_acc = net.evaluate(test_data, test_labels)
    
    arch_str = ' -> '.join(map(str, arch))
    print(f"{arch_str:25s}: Test accuracy = {test_acc:.3f}")

---

## Summary

### Advantages of the Single Class Approach

1. **Encapsulation**: All network state is contained in one object
2. **Clean interface**: `network.fit()`, `network.predict()`, `network.evaluate()`
3. **Easy to use**: Create network, train, evaluate - three steps
4. **Self-contained**: No need to pass weights/biases around

### Disadvantages

1. **Does too much**: Violates Single Responsibility Principle
2. **Hard to extend**: Adding new optimisers means modifying the class
3. **Hard to test**: Can't test components in isolation
4. **Hidden complexity**: All the details are buried in one place

### Design Choices Made

| Choice | Alternative | Rationale |
|--------|-------------|----------|
| Static methods for activations | Instance methods | Don't need `self`, pure functions |
| Xavier initialisation | He / random | Good default, works for both sigmoid and ReLU |
| Return history from `fit()` | Callbacks | Simpler, sufficient for basic use |
| Store layer_sizes | Infer from weights | Easier architecture inspection |
| Private methods with `_` prefix | No prefix | Convention signals internal use |

### For MNIST

```python
# For actual MNIST classification:
network = NeuralNetwork(
    layer_sizes=[784, 128, 64, 10],
    learning_rate=0.01,
    hidden_activation='relu'
)
network.fit(train_images, train_labels, epochs=10)
accuracy = network.evaluate(test_images, test_labels)
```

---

**Next notebook**: See how we can split this into separate, focused classes.